# 0. Imports

In [91]:
# Libraries
import os
import pandas as pd
import numpy as np
import datetime as dt

from datetime import date
from dataclasses import dataclass, field
from dateutil.relativedelta import relativedelta, WE

In [ ]:
# Global variables
DIR_PATH = os.getcwd()
DATA_PATH = os.path.join(DIR_PATH, 'market_data_split.csv')

t0 = dt.datetime(2026, 8, 19)

In [13]:
# Data
df = pd.read_csv(DATA_PATH, parse_dates=['maturity']) # SOFR quotes
display(df)

,type,tenor,quote,maturity
0,SOFR,ON,3.620,2026-08-20
1,SR1,Aug-26,96.361,2026-09-01
2,SR1,Sep-26,96.301,2026-10-01
3,SR3,Sep-26,96.195,2026-12-16
4,SR3,Dec-26,96.072,2027-03-17
5,SR3,Mar-27,96.015,2027-06-16
6,SR3,Jun-27,95.999,2027-09-15
7,OIS,2Y,4.000,2028-08-21
8,OIS,3Y,4.000,2029-08-20
9,OIS,5Y,4.040,2031-08-19


# 1. Bootstrapping

## 1.1 Discount curve with exact method

In [54]:
# Date functions

def act360(d1, d2):
    return (d2 - d1).days/360

def act365(d1, d2):
    return (d2 - d1).days/365

def following(d):
    d = np.datetime64(d, 'D')
    return np.busday_offset(d, offsets=0 ,roll='following')

def annual_schedule(t0, n_years):
    delta = relativedelta(years = n_years)
    T = t0 + delta
    return following(T)

In [ ]:
# Check sanity of annual_schedule

ois = df[df['type'] == 'OIS']
ois_years = ois["tenor"].str.rstrip("Y").astype(int) 

for n, expected in zip(ois_years, ois["maturity"]):
    sched = annual_schedule(t0, n)
    print(sched)
    

2028-08-21
2029-08-20
2031-08-19
2033-08-19
2036-08-19
2041-08-19
2046-08-20
2056-08-21


In [ ]:
@dataclass
class Instrument:
    type: str
    tenor: str
    quote: float
    start: date
    end: date

@dataclass
class OIS(Instrument):
    pay_dates: list[date]

@dataclass(frozen=True, eq=False)
class Curve:
    def 

In [92]:
def build_instrument(df, t0) -> list[Instrument]:

    insts = []

    for _, row in df.iterrows():

        if row.type == 'SOFR':
            inst = Instrument(row.type, row.tenor, row.quote, t0, row.maturity)

        elif row.type == 'SR1':
            delta = relativedelta(months = -1)
            start = row.maturity + delta
            if start < t0:

                start = t0
            inst = Instrument(row.type, row.tenor, row.quote, start, row.maturity)

        elif row.type == 'SR3':
            delta = relativedelta(months = -3, day = 1, weekday=WE(+3))
            start = row.maturity + delta
            inst = Instrument(row.type, row.tenor, row.quote, start, row.maturity)

        elif row.type == 'OIS':
            tenor_strip = int(row.tenor.rstrip('Y'))
            inst = OIS(row.type, row.tenor, row.quote, t0, row.maturity, 
                       pay_dates=[annual_schedule(t0, i) for i in range(1, tenor_strip + 1)]
                       )

        insts.append(inst)

    return insts

In [93]:
build_instrument(df, t0)

[Instrument(type='SOFR', tenor='ON', quote=3.62, start=datetime.datetime(2026, 8, 19, 0, 0), end=Timestamp('2026-08-20 00:00:00')),
 Instrument(type='SR1', tenor='Aug-26', quote=96.361, start=datetime.datetime(2026, 8, 19, 0, 0), end=Timestamp('2026-09-01 00:00:00')),
 Instrument(type='SR1', tenor='Sep-26', quote=96.301, start=Timestamp('2026-09-01 00:00:00'), end=Timestamp('2026-10-01 00:00:00')),
 Instrument(type='SR3', tenor='Sep-26', quote=96.195, start=Timestamp('2026-09-16 00:00:00'), end=Timestamp('2026-12-16 00:00:00')),
 Instrument(type='SR3', tenor='Dec-26', quote=96.072, start=Timestamp('2026-12-16 00:00:00'), end=Timestamp('2027-03-17 00:00:00')),
 Instrument(type='SR3', tenor='Mar-27', quote=96.015, start=Timestamp('2027-03-17 00:00:00'), end=Timestamp('2027-06-16 00:00:00')),
 Instrument(type='SR3', tenor='Jun-27', quote=95.999, start=Timestamp('2027-06-16 00:00:00'), end=Timestamp('2027-09-15 00:00:00')),
 OIS(type='OIS', tenor='2Y', quote=4.0, start=datetime.datetime(20